# Summary

- **Few missing values overall**, with Halifax, Nova Scotia concentrating all of them;
- **Linear regression fares better than ARIMA**, but worth considering biased results given that predictor $total = house + land$ and target variable is $house$.

In [ ]:
import pandas as pd
import geopandas as gpd
import os
import random
import numpy as np
import matplotlib.pyplot as plt

from pmdarima import auto_arima
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
df = pd.read_csv("data_cleaning\data\cleaned\stats_can\cleaned_data.csv")

gdf = gpd.read_file("data_cleaning\data\cleaned\cma_boundary\lcma000b21a_e.shp")


In [ ]:
print(gdf.head()) #inspect structure of shp file

In [ ]:
print(df.head()) #inspect structure of shp file

In [ ]:
# Dissemination Geography Unique Identifier (DGUID) in datasets change in the first part, the year

df["guid_core"] = df["dguid"].str[4:] 
gdf["guid_core"] = gdf["DGUID"].str[4:] 

common_dguid = set(df["guid_core"]).intersection(set(gdf["guid_core"]))

print("#DGUID in common:", len(common_dguid)/len(set(df["guid_core"]))) #~59% match between these two datasets

# Modelling - Prepping

In [ ]:
#Regions to show results: top 5 most populous ones & 5 random ones

random.seed(128)

top5 = [
    "Toronto, Ontario",
    "Montréal, Quebec",
    "Vancouver, British Columbia",
    "Ottawa-Gatineau, Ontario part, Ontario/Quebec",
    "Calgary, Alberta"
]

other_geos = [g for g in df["geo"].unique() if g not in top5]
random5 = random.sample(other_geos, 5)

In [ ]:
# Prepare data for modelling

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# Count NAs for each column
print((df.isnull().sum()/len(df))*100) #not much, 0.22% at total and house 

#Understand which regions have NA

print(df.groupby("geo").apply(lambda x: x.isna().sum())) #Halifax concentrates all NA!

#remove rows with NA
df = df.dropna()

## Prediction function

In [ ]:
def evaluate_and_plot(model_info, geo_name, method="ARIMAX"):
    # Get the dguid for this geo
    g = df[df["geo"] == geo_name]["dguid"].iloc[0]

    # Get the dictionary with model results and other specs
    info = model_info[g]

    model = info["model"]
    test  = info["test"]

    if len(test) == 0:
        print(f"No test data available for {geo_name}")
        return

    # Select method and predict
    if method == "ARIMAX":
        X_test = test[["total", "land"]]
        y_test = test["house"]
        pred = model.predict(n_periods=len(test), exogenous=X_test)

        # Print ARIMAX model details
        print(f"\n=== ARIMAX model for {geo_name} ===")
        print(f"Order (p,d,q): {model.order}")
        print("Coefficients:")
        print(model.params())

    elif method == "ARIMA":

        X_test = test[["total", "land"]]
        y_test = test["house"]
        pred = model.predict(n_periods=len(test), exogenous=X_test)

        # Print ARIMA model details
        print(f"\n=== ARIMA model for {geo_name} ===")
        print(f"Order (p,d,q): {model.order}")
        print("Coefficients:")
        print(model.params())        
         

    elif method == "Linear Regression with lag": 
        X_test = test[["total_lag1", "land_lag1"]]
        y_test = test["house"]
        pred = model.predict(X_test)

        # Print details of Linear Regression model with lag predictors
        print(f"\n=== Linear Regression  with lag, for {geo_name} ===")
        print("Intercept:", model.intercept_)
        print("Coefficients:")
        for name, coef in zip(["total_lag1", "land_lag1"], model.coef_):
            print(f"  {name}: {coef:.4f}")

    elif method == "Linear Regression":
        X_test = test[["total", "land"]]
        y_test = test["house"]

        pred = model.predict(X_test)

        # Print details of Linear Regression model
        print(f"\n=== Linear Regression model for {geo_name} ===")
        print("Intercept:", model.intercept_)
        print("Coefficients:")
        for name, coef in zip(["total", "land"], model.coef_):
            print(f"  {name}: {coef:.4f}")


    
    # Compute metrics
    pred = np.asarray(pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)

    # SMAPE: safe version of MAPE
    smape = 100 * np.mean(
        np.abs(y_test - pred) / ((np.abs(y_test) + np.abs(pred)) / 2)
    )

    # MDA (Mean directional accuracy) - quantifies number of times model correctly predicts the percentage of time a model correctly predicts the direction of a time series

    direction_actual = np.sign(np.diff(y_test.values, prepend=y_test.values[0]))
    direction_pred   = np.sign(np.diff(pred, prepend=pred[0]))
    mda = np.mean(direction_actual == direction_pred)

    print(f"\nPerformance for {geo_name}:")
    print(f"RMSE:  {rmse:.3f}")
    print(f"MAE:   {mae:.3f}")
    print(f"SMAPE: {smape:.2f}%")
    print(f"MDA:   {mda:.3f}")


    # Plot predicted and actual results
    plt.figure(figsize=(10,5))
    plt.plot(test["date"], y_test, label="Actual", linewidth=2)
    plt.plot(test["date"], pred, label="Predicted", linestyle="--")
    plt.title(f"{method} Forecast vs Actual – {geo_name}")
    plt.xlabel("Date")
    plt.ylabel("House Price Index")
    plt.legend()
    plt.grid(True)
    plt.show()


# ARIMA, per region

In [ ]:
arima_results_dict = {}

for g in df["dguid"].unique():
    sub = df[df["dguid"] == g].copy()
    sub = sub.sort_values("date")

    # Temporal train/test split (train until end of 2024)
    train = sub[sub["date"] <= "2024-12-31"]
    test  = sub[sub["date"] >  "2024-12-31"]

    # Separate target and regressors
    y_train = train["house"]
    X_train = train["house"]

    # Fit auto-ARIMA only on the training set
    model = auto_arima(
        y_train,
        exogenous=X_train,
        seasonal=False,
        stepwise=True,
        trace=False,
        error_action="ignore",
        suppress_warnings=True
    )

    # Store model and data for later evaluation and forecasting
    arima_results_dict[g] = {
        "model": model,
        "train": train,
        "test": test,
        "last_row": train.iloc[-1]
    }

    print(f"Model trained for {g} → order {model.order}")


## Evaluation

In [ ]:
print("=== TOP 5 MOST POPULATED CMAs ===")
for geo in top5:
    evaluate_and_plot(model_info = arima_results_dict, geo = geo, method = "ARIMA")

print("\n=== 5 RANDOM CMAs ===")
for geo in random5:
    evaluate_and_plot(model_info = arima_results_dict, geo = geo, method = "ARIMA")


## ARIMAX, per region

In [ ]:
arimax_results_dict = {}

for g in df["dguid"].unique():
    sub = df[df["dguid"] == g].copy()
    sub = sub.sort_values("date")

    # Temporal train/test split (train until end of 2024)
    train = sub[sub["date"] <= "2024-12-31"]
    test  = sub[sub["date"] >  "2024-12-31"]

    # Separate target and regressors
    y_train = train["house"]
    X_train = train[["total", "land"]]

    # Fit auto-ARIMA only on the training set
    model = auto_arima(
        y_train,
        exogenous=X_train,
        seasonal=False,
        stepwise=True,
        trace=False,
        error_action="ignore",
        suppress_warnings=True
    )

    # Store model and data for later evaluation and forecasting
    arimax_results_dict[g] = {
        "model": model,
        "train": train,
        "test": test,
        "last_row": train.iloc[-1]
    }

    print(f"Model trained for {g} → order {model.order}")




## Evaluation

In [ ]:
print("=== TOP 5 MOST POPULATED CMAs ===")
for geo in top5:
    evaluate_and_plot(model_info = arimax_results_dict, geo = geo)

print("\n=== 5 RANDOM CMAs ===")
for geo in random5:
    evaluate_and_plot(model_info = arimax_results_dict, geo = geo)


# Linear Regression with lag, per region

In [ ]:
lagged_lin_reg_results_dict = {}

for g in df["dguid"].unique():
    sub = df[df["dguid"] == g].copy()
    sub = sub.sort_values("date")

    # Create lagged predictors (to avoid data leakage)
    sub["house_lag1"] = sub["house"].shift(1)
    sub["total_lag1"] = sub["total"].shift(1)
    sub["land_lag1"] = sub["land"].shift(1)

    # Remove rows with missing values (due to lagging)
    sub = sub.dropna()

    # Temporal train/test split (train until end of 2024)
    train = sub[sub["date"] <= "2024-12-31"]
    test  = sub[sub["date"] >  "2024-12-31"]

    # Separate target and regressors
    y_train = train["house"]
    X_train = train[["total_lag1", "land_lag1"]]

    # Fit Linear Regression on the training set
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Store model and data for later evaluation and forecasting
    lagged_lin_reg_results_dict[g] = {
        "model": model,
        "train": train,
        "test": test
    }

    print(f"Linear Regression model with lag, trained for {g}")


## Evaluation

In [ ]:
print("=== TOP 5 MOST POPULATED CMAs ===")
for geo in top5:
    evaluate_and_plot(model_info = lagged_lin_reg_results_dict, geo = geo, method = "Linear Regression with lag")

print("\n=== 5 RANDOM CMAs ===")
for geo in random5:
    evaluate_and_plot(model_info = lagged_lin_reg_results_dict, geo = geo, method = "Linear Regression with lag")

# Linear Regression, per region

In [ ]:
lin_reg_results_dict = {}

for g in df["dguid"].unique():
    sub = df[df["dguid"] == g].copy()
    sub = sub.sort_values("date")

    # Temporal train/test split (train until end of 2024)
    train = sub[sub["date"] <= "2024-12-31"]
    test  = sub[sub["date"] >  "2024-12-31"]

    # Separate target and regressors
    y_train = train["house"]
    X_train = train[["total", "land"]]

    # Fit Linear Regression on the training set
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Store model and data for later evaluation and forecasting
    lin_reg_results_dict[g] = {
        "model": model,
        "train": train,
        "test": test
    }

    print(f"Linear Regression model trained for {g}")


## Evaluation

In [ ]:
print("=== TOP 5 MOST POPULATED CMAs ===")
for geo in top5:
    evaluate_and_plot(model_info = lagged_lin_reg_results_dict, geo = geo, method = "Linear Regression")

print("\n=== 5 RANDOM CMAs ===")
for geo in random5:
    evaluate_and_plot(model_info = lagged_lin_reg_results_dict, geo = geo, method = "Linear Regression")